# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1
...,...,...
2857,I don't have to respect your beliefs.||I only ...,0
2858,Women getting hit on by married managers at @u...,1
2859,@user no but i followed you and i saw you post...,0
2860,@user I dont know what it is but I'm in love y...,0


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1. Embedding lookup
        embedded = self.embedding(input_ids)

        # 2. LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Extract the final hidden state
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4. Apply dropout
        hidden = self.dropout(hidden)  

        # 5. Final classification layer (returns logits)
        output = self.fc(hidden)

        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = 1
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    """
    One epoch of training. 
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels' in each batch.
    - optimizer, criterion: training components (e.g., Adam, BCEWithLogitsLoss).
    - device: 'cpu' or 'cuda'.
    """
    model.train()
    losses = []
    correct_predictions = 0

    # For calculating precision, recall, F1:
    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 1) Forward pass -> raw logits
        logits = model(input_ids)  # shape: (batch_size, 1)
        logits = logits.squeeze(dim=1)  # shape: (batch_size,)

        # 2) Compute loss (BCEWithLogitsLoss expects raw logits)
        loss = criterion(logits, labels.float())

        # 3) Backprop + optimization
        loss.backward()
        optimizer.step()

        # 4) Track loss
        losses.append(loss.item())

        # 5) Convert logits -> probabilities -> predicted classes
        probs = torch.sigmoid(logits)          # in [0, 1]
        preds_cls = (probs >= 0.5).long()      # threshold at 0.5

        # 6) Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # 7) Collect for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate overall metrics for the epoch
    precision = precision_score(all_labels, all_preds, zero_division=0, average='macro')
    recall = recall_score(all_labels, all_preds, zero_division=0, average='macro')
    f1 = f1_score(all_labels, all_preds, zero_division=0, average='macro')
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1


def eval_model(model, data_loader, criterion, device):
    """
    Evaluation function. Similar to train_epoch, but no backprop.
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels'.
    - criterion: e.g., BCEWithLogitsLoss for binary classification.
    - device: 'cpu' or 'cuda'.
    """
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            # 1) Forward pass -> logits
            logits = model(input_ids)  # shape: (batch_size, 1)
            logits = logits.squeeze(dim=1)  # shape: (batch_size,)

            # 2) Compute loss
            loss = criterion(logits, labels.float())
            losses.append(loss.item())

            # 3) Convert logits -> probabilities -> predicted classes
            probs = torch.sigmoid(logits)
            preds_cls = (probs >= 0.5).long()

            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    # Metrics
    precision = precision_score(all_labels, all_preds, zero_division=0,average='macro')
    recall = recall_score(all_labels, all_preds, zero_division=0,average='macro')
    f1 = f1_score(all_labels, all_preds, zero_division=0,average='macro')
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1

# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device)
        
        val_acc, val_loss, val_prec, val_rec, val_f1 = eval_model(
            model, val_loader, criterion, device)
        
        print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, '
            f'Precision: {train_prec:.4f}, Recall: {train_rec:.4f}, F1 Score: {train_f1:.4f}')
        
        print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, '
            f'Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1 Score: {val_f1:.4f}')
    return train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1

In [10]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=['seed', 'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1',
                                'val_loss', 'val_acc', 'val_prec', 'val_rec', 'val_f1',
                                'test_loss', 'test_acc', 'test_prec', 'test_rec', 'test_f1',
                                'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
                                'max_memory_usage_test', 'max_vram_usage_test', 'total_time_test'])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1 = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1 = retval

    results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
                                                val_loss, val_acc, val_prec, val_rec, val_f1,
                                                test_loss, test_acc, test_prec, test_rec, test_f1,
                                                max_memory_usage_train, max_vram_usage_train, total_time_train,
                                                max_memory_usage_test, max_vram_usage_test, total_time_test]],
                                                columns=results.columns)], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6744, Accuracy: 0.5734, Precision: 0.5749, Recall: 0.5725, F1 Score: 0.5695
Val Loss: 0.6823, Accuracy: 0.5927, Precision: 0.5913, Recall: 0.5903, F1 Score: 0.5901
Epoch 2/5
Train Loss: 0.6187, Accuracy: 0.6544, Precision: 0.6559, Recall: 0.6539, F1 Score: 0.6531
Val Loss: 0.6612, Accuracy: 0.5906, Precision: 0.5892, Recall: 0.5880, F1 Score: 0.5877
Epoch 3/5
Train Loss: 0.5378, Accuracy: 0.7289, Precision: 0.7290, Recall: 0.7290, F1 Score: 0.7289
Val Loss: 0.7550, Accuracy: 0.5770, Precision: 0.6038, Recall: 0.5864, F1 Score: 0.5623
Epoch 4/5
Train Loss: 0.4498, Accuracy: 0.7900, Precision: 0.7900, Recall: 0.7900, F1 Score: 0.7900
Val Loss: 0.7761, Accuracy: 0.6262, Precision: 0.6255, Recall: 0.6233, F1 Score: 0.6230
Epoch 5/5
Train Loss: 0.3294, Accuracy: 0.8567, Precision: 0.8571, Recall: 0.8569, F1 Score: 0.8567
Val Loss: 0.9333, Accuracy: 0.6021, Precision: 0.6071, Recall: 0.6053, F1 Score: 0.6013


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_8612\3438238464.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6755, Accuracy: 0.5653, Precision: 0.5658, Recall: 0.5647, F1 Score: 0.5632
Val Loss: 0.6766, Accuracy: 0.5613, Precision: 0.5742, Recall: 0.5680, F1 Score: 0.5545
Epoch 2/5
Train Loss: 0.6303, Accuracy: 0.6331, Precision: 0.6347, Recall: 0.6325, F1 Score: 0.6314
Val Loss: 0.6557, Accuracy: 0.6031, Precision: 0.6050, Recall: 0.5971, F1 Score: 0.5925
Epoch 3/5
Train Loss: 0.5784, Accuracy: 0.6936, Precision: 0.6940, Recall: 0.6938, F1 Score: 0.6935
Val Loss: 0.6803, Accuracy: 0.6042, Precision: 0.6080, Recall: 0.6069, F1 Score: 0.6038
Epoch 4/5
Train Loss: 0.4826, Accuracy: 0.7732, Precision: 0.7733, Recall: 0.7733, F1 Score: 0.7732
Val Loss: 0.7102, Accuracy: 0.6168, Precision: 0.6167, Recall: 0.6169, F1 Score: 0.6165
Epoch 5/5
Train Loss: 0.3683, Accuracy: 0.8463, Precision: 0.8464, Recall: 0.8464, F1 Score: 0.8463
Val Loss: 0.8785, Accuracy: 0.6000, Precision: 0.5987, Recall: 0.5976, F1 Score: 0.5974


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6728, Accuracy: 0.5643, Precision: 0.5646, Recall: 0.5637, F1 Score: 0.5624
Val Loss: 0.6616, Accuracy: 0.5990, Precision: 0.5981, Recall: 0.5951, F1 Score: 0.5938
Epoch 2/5
Train Loss: 0.6194, Accuracy: 0.6474, Precision: 0.6474, Recall: 0.6474, F1 Score: 0.6474
Val Loss: 0.6757, Accuracy: 0.5780, Precision: 0.6037, Recall: 0.5872, F1 Score: 0.5642
Epoch 3/5
Train Loss: 0.5301, Accuracy: 0.7324, Precision: 0.7323, Recall: 0.7323, F1 Score: 0.7323
Val Loss: 0.6937, Accuracy: 0.6010, Precision: 0.5999, Recall: 0.5994, F1 Score: 0.5995
Epoch 4/5
Train Loss: 0.4277, Accuracy: 0.7994, Precision: 0.8001, Recall: 0.7997, F1 Score: 0.7994
Val Loss: 0.8420, Accuracy: 0.5864, Precision: 0.5952, Recall: 0.5913, F1 Score: 0.5836
Epoch 5/5
Train Loss: 0.3168, Accuracy: 0.8669, Precision: 0.8674, Recall: 0.8671, F1 Score: 0.8669
Val Loss: 0.8655, Accuracy: 0.5969, Precision: 0.5956, Recall: 0.5940, F1 Score: 0.5936


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_binary3.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,val_loss,val_acc,val_prec,val_rec,...,test_acc,test_prec,test_rec,test_f1,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.329382,0.856744,0.857093,0.856901,0.856735,0.933331,0.602094,0.607089,0.605349,...,0.617347,0.615036,0.620103,0.612237,1303.085938,231.628906,7.964942,1303.230469,194.671875,1.371358
1,3,0.368296,0.846261,0.846441,0.846377,0.846259,0.878453,0.600000,0.598717,0.597581,...,0.646684,0.631880,0.632852,0.632312,1303.589844,232.615234,6.463542,1303.589844,195.335938,1.332119
2,5,0.316768,0.866876,0.867355,0.867058,0.866863,0.865522,0.596859,0.595561,0.594008,...,0.642857,0.628658,0.630232,0.629287,1318.585938,231.113281,6.322849,1318.648438,194.156250,1.329296


In [12]:
torch.save(model.state_dict(), 'results/lstm_binary3.pth')